In [1]:
import os
import gc
import json
from typing import Any, Dict, List

import torch
from peft import PeftConfig, PeftModel
from tqdm.auto import tqdm
import evaluate

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

# ============================================================
# Config
# ============================================================

DATA_DIR = "dataset"
USE_FIELD = "sparql_dbpedia18"   # or "sparql_wikidata"
SPARQL_VARIANT = USE_FIELD.replace("sparql_", "")
OUTPUT_DIR = f"./qwen_sparql2text_encoder_rq1_{SPARQL_VARIANT}"

EVAL_SPLIT = "validation"        # "validation" or "test"
MAX_INPUT_LENGTH = 512
GEN_MAX_NEW_TOKENS = 64
BATCH_SIZE = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SAVE_PREDICTIONS = True
PRINT_EXAMPLES = 10


# ============================================================
# Utils
# ============================================================

def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_json(path: str) -> List[Dict]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(obj: Any, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def normalize_text(s: str) -> str:
    return " ".join(str(s).strip().split())


def safe_list(x: Any) -> List[str]:
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    s = str(x).strip()
    return [s] if s else []


def normalize_answer_item(x: str) -> str:
    return " ".join(str(x).strip().split())


def normalize_answer_list(answer: Any) -> List[str]:
    vals = safe_list(answer)
    return [normalize_answer_item(v) for v in vals if normalize_answer_item(v)]


# ============================================================
# Prompt
# ============================================================

def format_sparql_to_text_prompt(sparql: str) -> str:
    return (
        "You are an expert in knowledge graphs.\n\n"
        "### Instruction\n"
        "Convert the SPARQL query into a clear natural language question.\n\n"
        f"### SPARQL Query\n{sparql}\n\n"
        "### Natural Text\n"
    )


def clean_generated_question(text: str) -> str:
    text = normalize_text(text)

    if "### Natural Text" in text:
        text = text.split("### Natural Text", 1)[-1].strip()

    return normalize_text(text)


# ============================================================
# Data loading
# ============================================================

def load_examples(path: str, use_field: str) -> List[Dict[str, Any]]:
    raw = load_json(path)
    items = []

    for i, ex in enumerate(raw):
        sparql = ex.get(use_field)
        question = ex.get("question")
        answers = normalize_answer_list(ex.get("answer"))

        if sparql is None or not str(sparql).strip():
            continue
        if question is None or not str(question).strip():
            continue
        if len(answers) == 0:
            continue

        items.append(
            {
                "id": ex.get("id", i),
                "sparql": str(sparql).strip(),
                "question": normalize_text(question),
                "answer": answers,
            }
        )

    return items


# ============================================================
# Model loading
# ============================================================

def load_trained_model_and_tokenizer(lora_path: str):
    peft_cfg = PeftConfig.from_pretrained(lora_path)
    base_model_name = peft_cfg.base_model_name_or_path

    try:
        tokenizer = AutoTokenizer.from_pretrained(lora_path)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"

    if DEVICE == "cuda":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            quantization_config=bnb_config,
            device_map="auto",
        )
    else:
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            torch_dtype=torch.float32,
        ).to(DEVICE)

    model = PeftModel.from_pretrained(base_model, lora_path, is_trainable=False)
    model.eval()

    if hasattr(model, "generation_config"):
        try:
            model.generation_config.temperature = None
            model.generation_config.top_p = None
            model.generation_config.top_k = None
        except Exception:
            pass

    return model, tokenizer


# ============================================================
# Generation
# ============================================================

@torch.no_grad()
def generate_questions(
    model,
    tokenizer,
    examples: List[Dict[str, Any]],
    batch_size: int = 8,
    max_input_length: int = 512,
    max_new_tokens: int = 64,
) -> List[str]:
    all_predictions = []
    model_device = next(model.parameters()).device

    for start in tqdm(range(0, len(examples), batch_size), desc="Generating"):
        batch = examples[start:start + batch_size]
        prompts = [format_sparql_to_text_prompt(ex["sparql"]) for ex in batch]

        toks = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_length,
        )
        toks = {k: v.to(model_device) for k, v in toks.items()}

        generated = model.generate(
            **toks,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        decoded_full = tokenizer.batch_decode(generated, skip_special_tokens=True)

        for i, full_text in enumerate(decoded_full):
            prompt_text = normalize_text(prompts[i])
            full_text_norm = normalize_text(full_text)

            if full_text_norm.startswith(prompt_text):
                pred = full_text_norm[len(prompt_text):].strip()
            else:
                pred = full_text_norm.replace(prompt_text, "", 1).strip()

            pred = clean_generated_question(pred)
            all_predictions.append(pred)

    return all_predictions


# ============================================================
# ROUGE
# ============================================================

def compute_rouge_scores(predictions: List[str], references: List[str]) -> Dict[str, float]:
    rouge = evaluate.load("rouge")
    result = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True,
    )
    return {
        "rouge1": float(result["rouge1"]),
        "rouge2": float(result["rouge2"]),
        "rougeL": float(result["rougeL"]),
        "rougeLsum": float(result["rougeLsum"]),
    }


def print_examples(examples: List[Dict[str, Any]], predictions: List[str], n: int = 10):
    print("\n=== Sample Predictions ===")
    for i in range(min(n, len(examples))):
        print("\n" + "=" * 80)
        print(f"ID: {examples[i]['id']}")
        print(f"SPARQL:\n{examples[i]['sparql']}")
        print(f"\nReference: {examples[i]['question']}")
        print(f"Prediction: {predictions[i]}")


# ============================================================
# Main
# ============================================================

def main():
    cleanup_memory()

    eval_path = os.path.join(DATA_DIR, f"{EVAL_SPLIT}.json")
    examples = load_examples(eval_path, USE_FIELD)
    print(f"Loaded {len(examples)} examples from {eval_path}")

    model, tokenizer = load_trained_model_and_tokenizer(OUTPUT_DIR)

    predictions = generate_questions(
        model=model,
        tokenizer=tokenizer,
        examples=examples,
        batch_size=BATCH_SIZE,
        max_input_length=MAX_INPUT_LENGTH,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
    )

    references = [ex["question"] for ex in examples]
    rouge_scores = compute_rouge_scores(predictions, references)

    print("\n=== ROUGE Results ===")
    for k, v in rouge_scores.items():
        print(f"{k}: {v:.4f}")

    print_examples(examples, predictions, n=PRINT_EXAMPLES)

    if SAVE_PREDICTIONS:
        pred_rows = []
        for ex, pred in zip(examples, predictions):
            pred_rows.append(
                {
                    "id": ex["id"],
                    "sparql": ex["sparql"],
                    "reference_question": ex["question"],
                    "predicted_question": pred,
                    "answer": ex["answer"],
                }
            )

        pred_path = os.path.join(OUTPUT_DIR, f"{EVAL_SPLIT}_generated_predictions.json")
        score_path = os.path.join(OUTPUT_DIR, f"{EVAL_SPLIT}_rouge_scores.json")

        save_json(pred_rows, pred_path)
        save_json(rouge_scores, score_path)

        print(f"\nSaved predictions to: {pred_path}")
        print(f"Saved ROUGE scores to: {score_path}")


if __name__ == "__main__":
    main()

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql-emb/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 1526 examples from dataset/validation.json


Generating: 100%|██████████| 191/191 [12:44<00:00,  4.01s/it]



=== ROUGE Results ===
rouge1: 0.3707
rouge2: 0.1738
rougeL: 0.3518
rougeLsum: 0.3516

=== Sample Predictions ===

ID: 0
SPARQL:
SELECT ?answer WHERE { ?statement1 <http://www.w3.org/1999/02/22-rdf-syntax-ns#subject> <http://wikidata.dbpedia.org/resource/Q7313> . ?statement1 <http://www.w3.org/1999/02/22-rdf-syntax-ns#predicate> <http://www.wikidata.org/entity/P30>. ?statement1 <http://www.w3.org/1999/02/22-rdf-syntax-ns#object> ?X . ?statement2 <http://www.w3.org/1999/02/22-rdf-syntax-ns#subject> ?X. ?statement2 <http://www.w3.org/1999/02/22-rdf-syntax-ns#predicate> <http://www.wikidata.org/entity/P421> . ?statement2 <http://www.w3.org/1999/02/22-rdf-syntax-ns#object> ?answer . }

Reference: In what time was did the Yuan dynasty rule the Asian continent?
Prediction: What is the time zone of the continent of the Indian Ocean?

ID: 2
SPARQL:
select distinct ?obj where { ?statement <http://www.w3.org/1999/02/22-rdf-syntax-ns#subject> <http://wikidata.dbpedia.org/resource/Q217787> . ?stat